<a href="https://colab.research.google.com/github/lollopelle01/aac-mcp-agent/blob/main/eval/cpu/eval_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AACAgent: CPU Evaluation Notebook (llama.cpp / Colab)

Evaluation on standard CPU using `LlamaCppBackend` with Q4_K_M GGUF models.
Designed to run on **Google Colab with CPU runtime** (no GPU required),
to provide a uniform and reproducible hardware baseline.

**NOTE**: the backend is identical to the one used by the production app (`api/server.py`).

**Quick instructions:**
1. On Colab: Runtime → Change runtime type → **CPU**
2. Edit the `colab-env` cell with the desired parameters
3. Run all cells in order

In [1]:
##### COLAB ONLY — clone repo ##################################################
import subprocess, sys, os
import shutil

PROJECT_ROOT = "/content/aac-mcp-agent"

# Ensure we are in a known good directory before doing anything
os.chdir("/content/")

# Remove existing directory if it exists
if os.path.exists(PROJECT_ROOT):
    print(f"Removing existing directory: {PROJECT_ROOT}")
    shutil.rmtree(PROJECT_ROOT)

try:
    result = subprocess.run(
        ["git", "clone", "https://github.com/lollopelle01/aac-mcp-agent.git",
         PROJECT_ROOT],
        check=True,
        capture_output=True,
        text=True
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Error cloning repository: {e.cmd}")
    print(f"Return code: {e.returncode}")
    print(f"Output (stdout): {e.stdout}")
    print(f"Error (stderr): {e.stderr}")
    sys.exit(1) # Exit with an error code to signal failure

# Restore original behavior of changing into PROJECT_ROOT *after* cloning
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "app", "src"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "app"))

print("Repo cloned. CWD:", os.getcwd())


Repo cloned. CWD: /content/aac-mcp-agent


In [2]:
import os

# ENV VARS: edit here before running #####################################
os.environ["NB_MODELS"]            = "qwen2.5:3b"
os.environ["NB_N_ROWS"]            = "1"   # 0 = all 1760 sentences
os.environ["NB_LANG"]              = "en_eval"
os.environ["NB_MAX_RESULTS"]       = "0"     # 0 = use default from app/settings.py
os.environ["NB_SEED"]              = "42"
os.environ["NB_SPLIT_FILTER"]      = "all"   # "clear" | "vague" | "all"
os.environ["NB_N_THREADS"]         = "2"     # Colab CPU has 2 vCPUs
os.environ["NB_N_CTX"]             = "512"   # identical to production
os.environ["NB_OUTPUT_CSV"]        = "/content/aac-mcp-agent/eval/cpu/eval_cpu_colab.csv"
os.environ["NB_ANNOTATED_PARQUET"] = "/content/aac-mcp-agent/annotation/eval_final.parquet"

In [3]:
MODELS_RAW        = os.environ.get("NB_MODELS",               "qwen2.5:3b")
N_ROWS_ENV        = os.environ.get("NB_N_ROWS",               "100")
LANG_CODE         = os.environ.get("NB_LANG",                 "en_eval")
_max_results_env  = int(os.environ.get("NB_MAX_RESULTS",      "0"))
SEED              = int(os.environ.get("NB_SEED",             "42"))
SPLIT_FILTER      = os.environ.get("NB_SPLIT_FILTER",         "all")
N_THREADS         = int(os.environ.get("NB_N_THREADS",        "2"))
N_CTX             = int(os.environ.get("NB_N_CTX",            "512"))
ANNOTATED_PARQUET = os.environ.get("NB_ANNOTATED_PARQUET",    "/content/aac-mcp-agent/annotation/eval_final.parquet")
OUTPUT_CSV        = os.environ.get("NB_OUTPUT_CSV",           "/content/aac-mcp-agent/eval/cpu/eval_cpu_colab.csv")

MODELS = MODELS_RAW.split()
N_ROWS = int(N_ROWS_ENV)

print(f"Models            : {MODELS}")
print(f"N_rows            : {N_ROWS if N_ROWS > 0 else 'full dataset (1760)'}")
print(f"Seed              : {SEED}")
print(f"Lang              : {LANG_CODE}")
print(f"Max results       : {_max_results_env if _max_results_env > 0 else 'default (settings.py)'}")
print(f"Split filter      : {SPLIT_FILTER}")
print(f"n_threads (Colab) : {N_THREADS}")
print(f"n_ctx             : {N_CTX}")
print(f"Annotated parquet : {ANNOTATED_PARQUET}")
print(f"Output CSV        : {OUTPUT_CSV}")

Models            : ['qwen2.5:3b']
N_rows            : 1
Seed              : 42
Lang              : en_eval
Max results       : default (settings.py)
Split filter      : all
n_threads (Colab) : 2
n_ctx             : 512
Annotated parquet : /content/aac-mcp-agent/annotation/eval_final.parquet
Output CSV        : /content/aac-mcp-agent/eval/cpu/eval_cpu_colab.csv


In [4]:
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Base dependencies (without transformers / torch / bitsandbytes)
_pip(
    "pandas>=2.0",
    "pyarrow>=14",
    "tqdm>=4.66",
    "spacy>=3.7",
    "fastmcp",
    "pydantic>=2.0",
    "httpx>=0.24",
    "python-dotenv",
)

# NOTE: llama-cpp-python requires like 15 mins
try:
    import llama_cpp  # noqa: F401
    print("llama-cpp-python already installed.")
except ImportError:
    print("Installing llama-cpp-python (CPU wheel) ...")
    try:
        # First attempt: pre-built wheel (fast, ~30 seconds)
        _pip("llama-cpp-python")
    except subprocess.CalledProcessError:
        # Fallback: build from source without GPU accelerators (slow but works)
        print("Wheel not available — building from source (may take 5+ minutes) ...")
        env = os.environ.copy()
        env["CMAKE_ARGS"] = "-DLLAMA_CUBLAS=OFF -DLLAMA_METAL=OFF"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "llama-cpp-python", "--no-cache-dir"],
            env=env,
        )

# spaCy model
try:
    import spacy
    spacy.load("en_core_web_sm")
    print("spaCy model en_core_web_sm already present.")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

Installing llama-cpp-python (CPU wheel) ...
spaCy model en_core_web_sm already present.


In [5]:
from pathlib import Path

#### Download GGUFs from HuggingFace #########################################

# 1) GGUF files are not in the repo as they are too large, so we need to download
# them here directly from HuggingFace Hub and saved to app/models/.

# 2) The alias → (repo_id, filename) mapping is defined here because on Colab
# we cannot read settings.py before the path setup.

_MODELS_DIR = Path("/content/aac-mcp-agent/app/models")
_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Full mapping — update if new models are added to settings.py
_GGUF_SOURCES = {
    "qwen2.5:3b":    ("bartowski/Qwen2.5-3B-Instruct-GGUF",         "Qwen2.5-3B-Instruct-Q4_K_M.gguf"),
    "llama3.2:3b":   ("bartowski/Llama-3.2-3B-Instruct-GGUF",       "Llama-3.2-3B-Instruct-Q4_K_M.gguf"),
    "granite4:3b-h": ("bartowski/ibm-granite_granite-4.1-3b-GGUF",   "ibm-granite_granite-4.1-3b-Q4_K_M.gguf"),
    "mistral:7b":    ("bartowski/Mistral-7B-Instruct-v0.3-GGUF",     "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf"),
}

_pip("huggingface_hub>=0.22")
from huggingface_hub import hf_hub_download

downloaded: dict[str, str] = {}  # alias -> absolute path

for alias in MODELS:
    if alias not in _GGUF_SOURCES:
        print(f"'{alias}' has no GGUF source defined in _GGUF_SOURCES")
        continue
    repo_id, filename = _GGUF_SOURCES[alias]
    dest = _MODELS_DIR / filename
    if dest.exists():
        print(f"{alias}: already present ({dest.name})")
        downloaded[alias] = str(dest)
        continue
    print(f"{alias}: downloading {filename} from {repo_id} ...")
    path = hf_hub_download(
        repo_id   = repo_id,
        filename  = filename,
        local_dir = str(_MODELS_DIR),
    )
    downloaded[alias] = path
    print(f"saved to {path}")

missing = [m for m in MODELS if m not in downloaded]
if missing:
    raise RuntimeError(f"Missing GGUFs for: {missing}. Add them to _GGUF_SOURCES.")

qwen2.5:3b: downloading Qwen2.5-3B-Instruct-Q4_K_M.gguf from bartowski/Qwen2.5-3B-Instruct-GGUF ...


Qwen2.5-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

saved to /content/aac-mcp-agent/app/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf


In [6]:
PROJECT_ROOT = Path("/content/aac-mcp-agent")
APP          = PROJECT_ROOT / "app"
SRC          = APP / "src"

for p in [str(SRC), str(APP)]:
    if p not in sys.path:
        sys.path.insert(0, p)

EVAL_PARQUET = Path(ANNOTATED_PARQUET)

_out = Path(OUTPUT_CSV)
_out.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = _out

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"Eval parquet  : {EVAL_PARQUET}  exists={EVAL_PARQUET.exists()}")
print(f"Output CSV    : {OUTPUT_CSV_PATH}")

PROJECT_ROOT  : /content/aac-mcp-agent
Eval parquet  : /content/aac-mcp-agent/annotation/eval_final.parquet  exists=True
Output CSV    : /content/aac-mcp-agent/eval/cpu/eval_cpu_colab.csv


In [7]:
import ast
import csv
import logging
import time
from pathlib import Path

import pandas as pd

# Silence noisy libs — keep WARNING+ only
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for _noisy in ("httpx", "urllib3", "llama_cpp"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)

# Project imports
from config import AGENT_MAX_RESULTS
from settings import settings
from agent.agent import AACAgent, EvalContext
from agent.backends import LlamaCppBackend
from agent.session import SessionMemory
from mcp_server.models import Pictogram, Keyword
from mcp_server.tools.arasaac import get_pictogram_metadata
import mcp_server.tools.arasaac as _arasaac_mod

_arasaac_mod.LANG = LANG_CODE

EVAL_MAX_RESULTS = _max_results_env if _max_results_env > 0 else AGENT_MAX_RESULTS

# Route agent.run to file — captures [PLAN OUT], [PLAN], [CTX], [EVAL], [RESOLVE], [FALLBACK]
# setup_logging() is NOT called: it uses relative paths that don't exist on Colab.
_log_path = Path(OUTPUT_CSV_PATH).parent / "agent_run.log"
_fh = logging.FileHandler(_log_path, mode="w", encoding="utf-8")
_fh.setLevel(logging.INFO)
_fh.setFormatter(logging.Formatter("%(message)s"))
_agent_log = logging.getLogger("agent.run")
_agent_log.handlers.clear()   # remove stale handlers from previous runs
_agent_log.addHandler(_fh)
_agent_log.setLevel(logging.INFO)
_agent_log.propagate = False  # don't fall through to basicConfig (WARNING)

print(f"EVAL_MAX_RESULTS : {EVAL_MAX_RESULTS}")
print(f"agent.run log    : {_log_path}")

EVAL_MAX_RESULTS : 50
agent.run log    : /content/aac-mcp-agent/eval/cpu/agent_run.log


In [8]:
# CSV columns
CSV_COLUMNS = [
    "row_idx",
    "input_type",
    "turn_pos",
    "concept_text",
    "called_get_time",
    "called_get_schedule",
    "predicted_ids",
    "plan_method",
    "resolve_method",
    "planner_concepts",
]

In [9]:
import numpy as np

df_full = pd.read_parquet(EVAL_PARQUET)

# Parquet serializza le liste come numpy.ndarray — convertirle in list Python
# (isinstance str è un check legacy per CSV, non serve con parquet)
def _to_list(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, str):
        return ast.literal_eval(x)
    if isinstance(x, list):
        return x
    return []

df_full["concepts"] = df_full["concepts"].apply(_to_list)
df_full["schedule"] = df_full["schedule"].apply(_to_list)

print(f"Dataset: {len(df_full):,} rows  x  {list(df_full.columns)} columns")
# Verifica che schedule non sia più vuoto
n_with_sched = (df_full["schedule"].apply(len) > 0).sum()
print(f"Rows with schedule events: {n_with_sched:,} / {len(df_full):,}")

Dataset: 1,760 rows  x  ['sentence', 'concepts', 'caregiver_clear', 'caregiver_vague', 'time_of_day', 'event_time', 'schedule', 'tod_selection', 'split'] columns
Rows with schedule events: 1,760 / 1,760


In [10]:
_cap = N_ROWS if N_ROWS > 0 else 200   # more than 200 rows is infiseable on colab

df = (
    df_full
    .sample(min(_cap, len(df_full)), random_state=SEED)
    .reset_index()
    .rename(columns={"index": "row_idx"})
    .reset_index(drop=True)
)

print(f"Sample: {len(df)} rows (cap={_cap}, seed={SEED})")
print(f"row_idx range: {df['row_idx'].min()} - {df['row_idx'].max()}")

Sample: 1 rows (cap=1, seed=42)
row_idx range: 1125 - 1125


In [11]:
# ── Helper: gold metadata ─────────────────────────────────────────────────────

_gold_cache: dict[int, dict] = {}

def get_gold_meta(pic_id: int) -> dict:
    k = int(pic_id)
    if k not in _gold_cache:
        try:
            _gold_cache[k] = get_pictogram_metadata(pictogram_id=k, lang=LANG_CODE)
        except Exception:
            _gold_cache[k] = {}
    return _gold_cache[k]

def gold_as_pictogram(pic_id: int, concept: str) -> Pictogram:
    meta = get_gold_meta(pic_id)
    if meta:
        try:
            return Pictogram.model_validate(meta)
        except Exception:
            pass
    return Pictogram(id=int(pic_id), keywords=[Keyword(type=2, keyword=concept)])

def build_eval_ctx(row) -> EvalContext:
    from datetime import datetime, date, time as dtime
    event_time_str = str(row["event_time"])
    try:
        h, m = map(int, event_time_str.split(":"))
        current_dt = datetime.combine(date.today(), dtime(h, m)).isoformat()
    except Exception:
        current_dt = datetime.now().isoformat()

    mock_time = {
        "current_dt":  current_dt,
        "time_of_day": row["time_of_day"],
    }

    raw_sched = row["schedule"]
    if isinstance(raw_sched, np.ndarray):
        raw_sched = raw_sched.tolist()
    elif isinstance(raw_sched, str):
        raw_sched = ast.literal_eval(raw_sched)
    mock_schedule = raw_sched if isinstance(raw_sched, list) else []
    return EvalContext(mock_time=mock_time, mock_schedule=mock_schedule)

def teacher_force(agent: AACAgent, gold_id: int, concept: str) -> None:
    """
    Inject the gold pictogram into memory after each turn.
    """
    if not agent.memory.turns:
        return
    last = agent.memory.turns[-1]
    for t in last.topics:
        agent.memory.topic_frequency[t] = max(0, agent.memory.topic_frequency.get(t, 0) - 1)
    gold_pic    = gold_as_pictogram(gold_id, concept)
    gold_topics = SessionMemory.extract_topics([gold_pic])
    last.pictograms = [gold_pic]
    last.topics     = gold_topics
    for t in gold_topics:
        agent.memory.topic_frequency[t] = agent.memory.topic_frequency.get(t, 0) + 1


### Helper ############################################################################################
# NOTE: this is very useful if the runtime disconnects
class IncrementalCSV:
    def __init__(self, path: Path) -> None:
        self.path    = path
        self._is_new = not path.exists()
        self._buffer: list[dict] = []

    def add(self, rows: list[dict]) -> None:
        self._buffer.extend(rows)

    def flush(self) -> None:
        if not self._buffer:
            return
        mode = "w" if self._is_new else "a"
        with open(self.path, mode, newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
            if self._is_new:
                writer.writeheader()
                self._is_new = False
            writer.writerows(self._buffer)
        self._buffer.clear()


In [12]:
def run_multi_turn(agent: AACAgent, row: "pd.Series", input_type: str) -> list[dict]:
    """Run the multi-turn sequence for a row and an input_type.

    - Turn 0: real input (clear or vague) + FULL EvalContext with mock tools.
    - Turn 1+: input="" + EMPTY EvalContext (context already in session).
    - Teacher forcing: after each turn the gold is injected into memory.
    """
    concepts = row["concepts"]
    caregiver_input_t0 = str(row["caregiver_clear" if input_type == "clear" else "caregiver_vague"])

    agent.reset_session()
    results: list[dict] = []

    for turn_pos, concept_entry in enumerate(concepts):
        concept_text = concept_entry["concept_text"]
        gold_id      = int(concept_entry["gold_id"])

        if turn_pos == 0:
            ec        = build_eval_ctx(row)
            raw_input = caregiver_input_t0
        else:
            ec        = EvalContext()
            raw_input = ""

        window = agent.run(raw_input, eval_ctx=ec)

        predicted_ids    = [p.id for p in window]
        planner_concepts = [e["concept"] for e in agent.last_resolve_info]
        resolve_method   = [e["method"] for e in agent.last_resolve_info]

        results.append({
            "row_idx":             row['row_idx'], # Changed from row.name to row['row_idx']
            "input_type":          input_type,
            "turn_pos":            turn_pos,
            "concept_text":        concept_text,
            "called_get_time":     "get_time"     in ec.tool_calls,
            "called_get_schedule": "get_schedule" in ec.tool_calls,
            "predicted_ids":       str(predicted_ids),
            "plan_method":         agent.last_plan_method,
            "resolve_method":      resolve_method,
            "planner_concepts":    str(planner_concepts),
        })

        teacher_force(agent, gold_id, concept_text)

    return results

In [13]:
from tqdm.notebook import tqdm

SAVE_EVERY = 10

csv_writer = IncrementalCSV(OUTPUT_CSV_PATH)

for model_alias in MODELS:
    print(f"\n{'━'*70}\n  MODEL: {model_alias}\n{'━'*70}")

    gguf_path = downloaded.get(model_alias)
    if not gguf_path:
        print(f"   GGUF not available for '{model_alias}', skipping")
        continue

    backend = LlamaCppBackend(
        model_path  = gguf_path,
        n_ctx       = N_CTX,
        n_threads   = N_THREADS,
        temperature = 0.0,
        max_tokens  = 300, # before it was 150 but some inputs require more tokens
        verbose     = False,
    )

    agent = AACAgent(
        model          = model_alias,
        backend        = backend,
        lang           = LANG_CODE,
        max_results    = EVAL_MAX_RESULTS,
        fetch_schedule = False,   # mocked via EvalContext, no live calls
    )

    print("  Loading GGUF into RAM ...", flush=True)
    t_load = time.monotonic()
    agent.backend._ensure_loaded()
    print(f"  GGUF loaded in {time.monotonic() - t_load:.1f}s", flush=True)

    n_errors = 0
    pbar = tqdm(df.iterrows(), total=len(df), desc=model_alias)

    for _, row in pbar:
        split_val = str(row["split"])

        if split_val == "none":
            continue
        elif split_val == "clear":
            input_types = ["clear"]
        elif split_val == "vague":
            input_types = ["vague"]
        else:  # "both"
            input_types = ["clear", "vague"]

        if SPLIT_FILTER != "all":
            input_types = [t for t in input_types if t == SPLIT_FILTER]
        if not input_types:
            continue

        try:
            for input_type in input_types:
                row_results = run_multi_turn(agent, row, input_type)
                csv_writer.add(row_results)
            if pbar.n % SAVE_EVERY == 0:
                csv_writer.flush()
        except Exception as exc:
            n_errors += 1
            pbar.set_postfix(errors=n_errors)
            print(f"  [ERROR] row={row['row_idx']}: {exc}", flush=True)

    csv_writer.flush()
    agent.unload()
    print(f"  Model {model_alias!r} completed.")

print(f"\nDone. Output: {OUTPUT_CSV_PATH}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  MODEL: qwen2.5:3b
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Loading GGUF into RAM ...


llama_context: n_ctx_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  GGUF loaded in 8.9s


qwen2.5:3b:   0%|          | 0/1 [00:00<?, ?it/s]

  Model 'qwen2.5:3b' completed.

Done. Output: /content/aac-mcp-agent/eval/cpu/eval_cpu_colab.csv


In [14]:
res = pd.read_csv(OUTPUT_CSV_PATH)
for col in ("predicted_ids", "planner_concepts"):
    res[col] = res[col].apply(ast.literal_eval)

print(f"Total rows            : {len(res)}")
print(f"\ninput_type distribution:\n{res['input_type'].value_counts()}")

t0 = res[res["turn_pos"] == 0]
print(f"\nTool call rate at turn_pos==0 (by input_type):")
print(t0.groupby("input_type")[["called_get_time","called_get_schedule"]].mean())
print("EXPECTED: ~1.0 for vague, ~0.0 for clear")

print(f"\nplan_method distribution:\n{res['plan_method'].value_counts()}")
print(f"\nresolve_method distribution:\n{res['resolve_method'].value_counts()}")
print(f"\nAverage window size: {res['predicted_ids'].apply(len).mean():.1f}")
print(f"\nFirst 3 rows example:\n{res.head(3).to_string()}")

Total rows            : 6

input_type distribution:
input_type
clear    3
vague    3
Name: count, dtype: int64

Tool call rate at turn_pos==0 (by input_type):
            called_get_time  called_get_schedule
input_type                                      
clear                   0.0                  0.0
vague                   1.0                  1.0
EXPECTED: ~1.0 for vague, ~0.0 for clear

plan_method distribution:
plan_method
llm               4
fallback_empty    2
Name: count, dtype: int64

resolve_method distribution:
resolve_method
['exact', 'exact']                                                                                                                                                               2
['none', 'exact', 'none', 'exact', 'exact', 'exact', 'token']                                                                                                                    1
['none', 'exact', 'none', 'exact', 'exact', 'none', 'exact']                                    

In [15]:
df_full.iloc[1125]

,1125
sentence,I struggle with the math problems the most.
concepts,"[{'candidate_ids': [9851, 37810, 25133, 8643, ..."
caregiver_clear,He struggles with math problems at 14:30 this ...
caregiver_vague,he keeps fidgeting during lessons
time_of_day,afternoon
event_time,14:30
schedule,"[{'description': None, 'location': 'school', '..."
tod_selection,model
split,both
